In [2]:
from sentence_transformers import SentenceTransformer
from transformers import AutoModel
import sentence_transformers
import transformers


print(f"sentence_transformers version: {sentence_transformers.__version__}")
print(f"transformers version: {transformers.__version__}")

/root/kyx/conteb-research/conteb/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


sentence_transformers version: 5.3.0
transformers version: 5.3.0


In [4]:
sentences = ["", "This is an example sentence", "Each sentence is converted"]

model = SentenceTransformer('nishimoto/contriever-sentencetransformer')
embeddings = model.encode(sentences)
print(embeddings)
print(model)
print(f"#params: {sum(p.numel() for p in model.parameters()) / 1e6:.2f} M")


KeyboardInterrupt: 

In [10]:
sentences = ["", "This is an example sentence", "Each sentence is converted"]

model = SentenceTransformer('lightonai/modernbert-embed-large')
embeddings = model.encode(sentences)
print(embeddings)
print(model)
print("=" * 40)
print("SentenceTransformer 架构:")
for name, module in model.named_children():
    print(f"[{name}] {module.__class__.__name__}")
    print(module)

print("=" * 40)
print("底层 Transformer 架构:")
transformer = model[0].auto_model
print(transformer)

print("=" * 40)
print("ModernBERT 配置:")
print(transformer.config)

print("=" * 40)
print("各模块参数量:")
for name, module in transformer.named_modules():
    parameter_count = sum(parameter.numel() for parameter in module.parameters(recurse=False))
    if parameter_count:
        print(f"{name or '<root>':60s} {parameter_count / 1e6:10.2f} M")

print("=" * 40)
print(f"#params: {sum(parameter.numel() for parameter in model.parameters()) / 1e6:.2f} M")


Loading weights: 100%|██████████| 170/170 [00:00<00:00, 6734.08it/s]


[[ 3.25514413e-02 -7.89453334e-05  3.76247871e-03 ...  3.15815583e-02
   1.42119685e-02 -2.41722297e-02]
 [ 4.82675526e-03  1.59786176e-02  1.02230785e-02 ...  2.11760309e-02
   1.22410078e-02  4.09738757e-02]
 [ 3.61054093e-02  9.36497562e-03  6.01036614e-03 ...  2.03489326e-02
   2.64529069e-03  3.23007777e-02]]
SentenceTransformer(
  (0): Transformer({'max_seq_length': 8192, 'do_lower_case': False, 'architecture': 'ModernBertModel'})
  (1): Pooling({'word_embedding_dimension': 1024, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)
SentenceTransformer 架构:
[0] Transformer
Transformer({'max_seq_length': 8192, 'do_lower_case': False, 'architecture': 'ModernBertModel'})
[1] Pooling
Pooling({'word_embedding_dimension': 1024, 'pooling_mode_cls_token': False, 'pooling_mode_mea

In [ ]:
model_ctx = AutoModel.from_pretrained(
    "perplexity-ai/pplx-embed-context-v1-0.6B",
    trust_remote_code=True
)

doc_chunks = [
    [
        "Curiosity begins in childhood with endless questions about the world.",
        "As we grow, curiosity drives us to explore new ideas.",
        "Scientific breakthroughs often start with a curious question."
    ],
    [
        "The curiosity rover explores Mars searching for ancient life.",
        "Each discovery on Mars sparks new questions about the universe."
    ]
]
# Returns list of numpy arrays (one per document)
# embeddings[0].shape = (3, 1024), embeddings[1].shape = (2, 1024)
embeddings = model_ctx.encode(doc_chunks)

print(embeddings)
print(model_ctx)
print(f"#params: {sum(p.numel() for p in model_ctx.parameters()) / 1e6:.2f} M")


You are using a model of type `bidirectional_pplx_qwen3` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.
Loading weights: 100%|██████████| 310/310 [00:00<00:00, 4618.75it/s]


[array([[-26.       ,  -9.       ,   0.       , ...,   3.       ,
         -3.       , -26.       ],
       [-13.       ,  -2.       ,   0.       , ...,  14.       ,
         -4.       , -13.       ],
       [ -2.       , -14.       ,   0.       , ...,   9.       ,
          0.       ,  -1.9999999]], dtype=float32), array([[ 24., -32.,   0., ...,  -4., -21.,  -6.],
       [ 13.,  10.,   0., ...,   8., -15., -11.]], dtype=float32)]
PPLXQwen3ContextualModel(
  (embed_tokens): Embedding(151936, 1024)
  (layers): ModuleList(
    (0-27): 28 x Qwen3DecoderLayer(
      (self_attn): Qwen3Attention(
        (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
        (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
        (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
        (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
        (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
        (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
      

In [ ]:
model = SentenceTransformer(
    "perplexity-ai/pplx-embed-v1-0.6B",
    trust_remote_code=True
)

texts = [
    "",
    "Scientists explore the universe driven by curiosity.",
    "Children learn through curious exploration.",
    "Historical discoveries began with curious questions.",
    "Animals use curiosity to adapt and survive.",
    "Philosophy examines the nature of curiosity.",
]

embeddings = model.encode(texts) # Shape: (5, 1024), quantized to int8
print("embeddings (int8):", embeddings)

embeddings = model.encode(texts, quantization="binary") # Shape: (5, 1024), quantized to binary
print("embeddings (binary):", embeddings)

print(model)
print(f"#params: {sum(p.numel() for p in model.parameters()) / 1e6:.2f} M")

Loading weights: 100%|██████████| 310/310 [00:00<00:00, 5918.52it/s]


embeddings (int8): [[  0.   0.   0. ...   0.   0.   0.]
 [-21. -31.   0. ...  30.  -3. -21.]
 [-37.  27.   0. ...  22.   5. -15.]
 [ -8.  -6.   0. ...   5.  11. -33.]
 [-51.  26.   0. ...  46. -14. -14.]
 [  9. -18.   0. ...  57.  13.  -9.]]
embeddings (binary): [[ 1.          1.          1.         ...  1.          1.
   1.        ]
 [-1.         -1.         -0.99999994 ...  1.         -1.
  -1.        ]
 [-1.          1.         -1.         ...  1.          1.
  -0.99999994]
 [-0.99999994 -0.99999994 -1.         ...  1.          1.
  -1.        ]
 [-1.          0.99999994 -1.         ...  1.         -1.
  -1.        ]
 [ 1.         -0.99999994 -1.         ...  1.          1.
  -0.99999994]]
SentenceTransformer(
  (0): Transformer({'max_seq_length': 32768, 'do_lower_case': False, 'architecture': 'PPLXQwen3Model'})
  (1): Pooling({'word_embedding_dimension': 1024, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqr

: 